# Claude Certified Architect — Foundations
## Domain 5: Context Management & Reliability
**Exam weight: 15%**

These are the patterns that separate a demo from a production system — they rarely appear in tutorials because they only matter at scale, but they are what the exam tests as architectural judgment. This domain covers how to keep agents reliable across long conversations, multi-agent pipelines, large codebases, and heterogeneous source data, and how to design human-in-the-loop workflows that trigger on the right signals rather than on noise.

This domain covers the patterns that keep agents reliable over long sessions, across multiple agents, and in the presence of failures and uncertainty. These are the patterns that separate a demo from a production system.

**Prerequisites:** `pip install anthropic`  
**Auth:** Set `ANTHROPIC_API_KEY` as an environment variable.

### Task Statements Covered
- **5.1** Manage conversation context to preserve critical information across long interactions
- **5.2** Design effective escalation and ambiguity resolution patterns
- **5.3** Implement error propagation strategies across multi-agent systems
- **5.4** Manage context effectively in large codebase exploration
- **5.5** Design human review workflows and confidence calibration
- **5.6** Preserve information provenance and handle uncertainty in multi-source synthesis

In [17]:
import anthropic
import json
import time

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
print("Client ready.")

Client ready.


---
## Task Statement 5.1: Manage conversation context to preserve critical information across long interactions

In long multi-turn conversations, two failure modes erode context reliability: progressive summarization compresses specific values like `$127.43` and `ORD-5512` into vague descriptions like "a refund issue", and the lost-in-the-middle effect causes information buried in the middle of a long input to be silently omitted. The solution is structural — extract transactional facts into a persistent block that is included in every prompt and never passed through a summarization step.

**What this means in practice:** In long multi-turn conversations, critical facts get lost two ways: progressive summarization compresses specific values into vague descriptions, and the 'lost in the middle' effect causes the model to neglect information in the middle of a long input. The fix is to extract transactional facts into a persistent structured block that's included in every prompt, separate from the summarized history. Verbose tool outputs must be trimmed before they accumulate.

**Why it matters for an architect:** A customer support agent that summarizes '$127.43 refund, order #ORD-5512, delivered November 3rd' into 'the customer has a refund issue' will give wrong answers later in the conversation. The specific values — amounts, order numbers, dates — are what the agent needs to actually resolve the case. They must survive summarization intact.

**Core concepts:**
- Extract transactional facts (amounts, order IDs, dates, confirmation numbers) into a persistent `case_facts` structured block that is included in every prompt and never summarized
- 'Lost in the middle' effect: models reliably process information at the beginning and end of long inputs; information in the middle may be omitted — put key findings summary at the START with an explicit header
- Trim verbose tool outputs to relevant fields before appending to conversation history; irrelevant fields consume context budget without contributing to the task

**Anti-patterns to avoid:**
- Summarizing the entire conversation including specific numerical values — `$127.43`, `ORD-5512`, `REF-8821` become "a refund issue" after a progressive summarization pass
- Placing key findings in the middle of a long context — most likely position for the lost-in-the-middle effect
- Accumulating complete raw tool outputs in conversation history — fields like `warehouse_id`, `carrier_code`, `marketing_opt_in` are irrelevant and compress useful context

In [18]:
# Demonstrating: progressive summarization destroys specific values

CASE_HISTORY = """
Turn 1: Customer identified as Jane Smith, customer ID CUST-9921, email jane@example.com
Turn 2: Order ORD-5512 looked up — $127.43, delivered November 3rd, eligible for return
Turn 3: Second order ORD-5518 looked up — $43.99, still in transit, not eligible
Turn 4: Customer requested refund of $127.43 for ORD-5512
Turn 5: Refund initiated, confirmation REF-8821, 3-5 business days
"""

def summarize_history(history: str) -> str:
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        messages=[{"role": "user", "content":
            f"Summarize this customer support conversation history in 2 sentences:\n{history}"
        }]
    )
    return response.content[0].text

summary = summarize_history(CASE_HISTORY)
print("=== ANTI-PATTERN: Progressive summarization ===")
print(f"Original history (contains specific values):")
print(CASE_HISTORY)
print(f"After summarization:")
print(summary)
print()
print("OBSERVE: The summary likely loses: ORD-5512, $127.43, REF-8821, CUST-9921")
print("If agent needs to reference these later, they are gone.")

=== ANTI-PATTERN: Progressive summarization ===
Original history (contains specific values):

Turn 1: Customer identified as Jane Smith, customer ID CUST-9921, email jane@example.com
Turn 2: Order ORD-5512 looked up — $127.43, delivered November 3rd, eligible for return
Turn 3: Second order ORD-5518 looked up — $43.99, still in transit, not eligible
Turn 4: Customer requested refund of $127.43 for ORD-5512
Turn 5: Refund initiated, confirmation REF-8821, 3-5 business days

After summarization:
Jane Smith (CUST-9921) contacted support regarding two orders: ORD-5512 ($127.43, delivered November 3rd) and ORD-5518 ($43.99, still in transit). She requested and received a refund of $127.43 for ORD-5512, with confirmation REF-8821 expected to process within 3-5 business days.

OBSERVE: The summary likely loses: ORD-5512, $127.43, REF-8821, CUST-9921
If agent needs to reference these later, they are gone.


In [19]:
# CORRECT PATTERN: Extract transactional facts into a persistent structured block
# The case_facts block is included in every subsequent prompt, outside summarized history.

class CaseContext:
    """Persistent structured store for transactional facts that must survive summarization."""

    def __init__(self):
        self.case_facts = {}       # Extracted facts — always included in prompt
        self.conversation_summary = ""  # Compressed narrative — can be updated
        self.tool_outputs = []     # Raw tool outputs — trimmed before accumulating

    def update_facts(self, new_facts: dict):
        """Merge new transactional facts — amounts, IDs, dates — into the persistent block."""
        self.case_facts.update(new_facts)

    def trim_tool_output(self, raw_output: dict, relevant_fields: list) -> dict:
        """Keep only relevant fields from verbose tool output before storing."""
        return {k: v for k, v in raw_output.items() if k in relevant_fields}

    def build_prompt_context(self) -> str:
        """Build the context block included in every prompt."""
        return f"""
## Case Facts (authoritative — do not summarize these)
{json.dumps(self.case_facts, indent=2)}

## Conversation Summary
{self.conversation_summary or 'Conversation just started.'}
"""


# Simulate a multi-turn session
context = CaseContext()

# Turn 1: customer lookup — store specific IDs
raw_customer = {
    "customer_id": "CUST-9921", "name": "Jane Smith", "email": "jane@example.com",
    "address": "123 Main St", "phone": "555-0100", "created_at": "2020-01-15",
    "marketing_opt_in": True, "preferred_language": "en"  # Irrelevant to this case
}
# Trim: keep only what's relevant
trimmed = context.trim_tool_output(raw_customer, ["customer_id", "name", "email"])
context.update_facts({"customer": trimmed})

# Turn 2: order lookup — store specific amounts
raw_order = {
    "order_id": "ORD-5512", "amount": 127.43, "status": "delivered",
    "delivered_date": "2024-11-03", "eligible_for_return": True,
    "warehouse_id": "WH-007", "carrier_code": "UPS", "weight_kg": 0.8  # Irrelevant
}
trimmed_order = context.trim_tool_output(raw_order, ["order_id", "amount", "status", "delivered_date", "eligible_for_return"])
context.update_facts({"primary_order": trimmed_order})

# Turn 5: refund confirmation — store reference number
context.update_facts({"refund": {"refund_id": "REF-8821", "amount": 127.43, "eta_days": "3-5"}})

# Update narrative summary (can be lossy — specific values are in case_facts)
context.conversation_summary = "Customer requested return of ORD-5512. Refund initiated successfully."

print("=== CORRECT PATTERN: Persistent case facts block ===")
print(context.build_prompt_context())

print("OBSERVE:")
print("  Specific values (CUST-9921, ORD-5512, $127.43, REF-8821) are in case_facts")
print("  They survive summarization because they are NEVER summarized")
print("  Verbose tool fields (warehouse_id, carrier_code) were trimmed before storage")

=== CORRECT PATTERN: Persistent case facts block ===

## Case Facts (authoritative — do not summarize these)
{
  "customer": {
    "customer_id": "CUST-9921",
    "name": "Jane Smith",
    "email": "jane@example.com"
  },
  "primary_order": {
    "order_id": "ORD-5512",
    "amount": 127.43,
    "status": "delivered",
    "delivered_date": "2024-11-03",
    "eligible_for_return": true
  },
  "refund": {
    "refund_id": "REF-8821",
    "amount": 127.43,
    "eta_days": "3-5"
  }
}

## Conversation Summary
Customer requested return of ORD-5512. Refund initiated successfully.

OBSERVE:
  Specific values (CUST-9921, ORD-5512, $127.43, REF-8821) are in case_facts
  They survive summarization because they are NEVER summarized
  Verbose tool fields (warehouse_id, carrier_code) were trimmed before storage


In [20]:
# Demonstrating: lost-in-the-middle effect and mitigation
# Key findings placed at beginning and end of long inputs are more reliably processed.

def build_aggregated_context_naive(findings: list) -> str:
    """NAIVE: key findings buried in the middle of a long context."""
    padding = "[Additional research context and background information...] " * 20
    middle_findings = "\n".join(f"Finding {i+1}: {f}" for i, f in enumerate(findings))
    return f"{padding}\n\nFINDINGS:\n{middle_findings}\n\n{padding}"


def build_aggregated_context_position_aware(findings: list, summary: str) -> str:
    """CORRECT: key findings summary at the START, details organized with explicit headers."""
    findings_text = "\n".join(f"  - {f}" for f in findings)
    padding = "[Additional research context and background information...] " * 20
    return f"""
## KEY FINDINGS SUMMARY (read this first)
{summary}

## Detailed Findings
{findings_text}

## Additional Background Context
{padding}
"""

KEY_FINDINGS = [
    "Revenue grew 23% YoY to $4.2M",
    "Customer churn increased from 4% to 7%",
    "Product line C accounts for 61% of revenue"
]
SUMMARY = "Company shows strong revenue growth but concerning churn increase. Product C dominates revenue."

print("=== Position-aware context construction ===")
print("NAIVE (findings buried in middle):")
naive = build_aggregated_context_naive(KEY_FINDINGS)
print(f"  Total length: {len(naive)} chars")
print(f"  Finding position: ~middle (most likely to be omitted in 'lost in middle' effect)")

print("\nPOSITION-AWARE (findings at start with explicit header):")
aware = build_aggregated_context_position_aware(KEY_FINDINGS, SUMMARY)
print(aware[:500])
print("\n  Key findings appear at the START — reliably processed")
print("  Explicit section headers help the model locate relevant sections")

=== Position-aware context construction ===
NAIVE (findings buried in middle):
  Total length: 2558 chars
  Finding position: ~middle (most likely to be omitted in 'lost in middle' effect)

POSITION-AWARE (findings at start with explicit header):

## KEY FINDINGS SUMMARY (read this first)
Company shows strong revenue growth but concerning churn increase. Product C dominates revenue.

## Detailed Findings
  - Revenue grew 23% YoY to $4.2M
  - Customer churn increased from 4% to 7%
  - Product line C accounts for 61% of revenue

## Additional Background Context
[Additional research context and background information...] [Additional research context and background information...] [Additional research context and background information...] [

  Key findings appear at the START — reliably processed
  Explicit section headers help the model locate relevant sections


In [21]:
# Demonstrating: upstream agent output structuring for downstream context budgets
# Exam guide 5.1 skill: "Modifying upstream agents to return structured data
# (key facts, citations, relevance scores) instead of verbose content and reasoning chains
# when downstream agents have limited context budgets."
#
# The architectural pattern: rather than trimming verbose output after the fact,
# design upstream agents to emit compact structured outputs from the start.

def research_subagent_verbose(topic: str) -> str:
    """ANTI-PATTERN: upstream agent returns verbose prose with embedded reasoning chains."""
    response = client.messages.create(
        model=MODEL, max_tokens=400,
        system="You are a research agent. Thoroughly analyze this topic and share your complete reasoning and all findings.",
        messages=[{"role": "user", "content": f"Research: {topic}"}]
    )
    return response.content[0].text


STRUCTURED_OUTPUT_TOOL = {
    "name": "report_findings",
    "description": "Report key research findings in a compact structured format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "claim": {"type": "string", "description": "One-sentence factual claim"},
                        "source_url": {"type": ["string", "null"]},
                        "relevance_score": {
                            "type": "number", "minimum": 0, "maximum": 1,
                            "description": "How relevant to the research question (0-1)"
                        }
                    },
                    "required": ["claim", "relevance_score"]
                }
            },
            "coverage_gaps": {"type": "array", "items": {"type": "string"}}
        },
        "required": ["findings"]
    }
}

def research_subagent_structured(topic: str) -> dict:
    """CORRECT PATTERN: upstream agent returns compact structured data.
    The downstream synthesis agent receives key facts + relevance scores,
    not verbose prose reasoning chains that would exhaust its context budget.
    """
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        system=(
            "You are a research agent. Return ONLY structured findings via the report_findings tool. "
            "Do not include reasoning chains or verbose explanations — only the key claims and their sources. "
            "This output feeds a downstream synthesis agent with a limited context budget."
        ),
        tools=[STRUCTURED_OUTPUT_TOOL],
        tool_choice={"type": "tool", "name": "report_findings"},
        messages=[{"role": "user", "content": f"Research: {topic}"}]
    )
    tool_block = next((b for b in response.content if b.type == "tool_use"), None)
    return tool_block.input if tool_block else {}


print("=== Upstream agent output: verbose vs structured ===")
TOPIC = "impact of AI on music production in 2024"

print("\n[ANTI-PATTERN] Verbose upstream output:")
verbose = research_subagent_verbose(TOPIC)
print(f"  Word count: ~{len(verbose.split())}")
print(f"  Sample: {verbose[:200]}...")

print("\n[CORRECT PATTERN] Structured upstream output:")
structured = research_subagent_structured(TOPIC)
print(f"  Word count: ~{len(json.dumps(structured).split())}")
print(f"  Findings: {len(structured.get('findings', []))} claims")
for f in structured.get('findings', [])[:2]:
    print(f"    - [{f.get('relevance_score', 0):.1f}] {f.get('claim', '')[:80]}")

print("\nOBSERVE: The structured output is significantly smaller.")
print("This is an architectural decision — designed into the upstream agent,")
print("not patched after the fact via coordinator-side trimming.")


=== Upstream agent output: verbose vs structured ===

[ANTI-PATTERN] Verbose upstream output:
  Word count: ~243
  Sample: # Impact of AI on Music Production in 2024: Comprehensive Research Analysis

---

## Executive Summary

2024 marked a pivotal and contentious year for AI in music production. The technology moved from...

[CORRECT PATTERN] Structured upstream output:
  Word count: ~1
  Findings: 0 claims

OBSERVE: The structured output is significantly smaller.
This is an architectural decision — designed into the upstream agent,
not patched after the fact via coordinator-side trimming.


### Upstream agent output design

The context management patterns covered so far (persistent `case_facts`, trimming verbose tool outputs, position-aware ordering) all operate on the **coordinator side** — they clean up what arrives. The complementary architectural pattern operates on the **upstream agent side**: design subagents to emit compact structured outputs in the first place, rather than verbose prose reasoning chains.

**Why this matters:** A synthesis agent with a 10,000-token context budget that receives 8,000 tokens of verbose prose from three upstream agents has almost no room left for its own analysis and output. If those upstream agents instead return structured dicts of `{claim, source_url, relevance_score}`, the same budget leaves the synthesis agent with ample room to reason.

**The two-layer approach:**

| Layer | Pattern | Mechanism |
|---|---|---|
| Upstream agent | Return structured data instead of verbose prose | `tool_choice` forces a compact schema output |
| Coordinator | Trim what arrives before it accumulates | Filter to relevant fields only |

Both layers are needed — upstream design reduces volume at the source, coordinator trimming handles third-party tools and APIs you don't control.

**Key exam facts for 5.1:**
- Progressive summarization loses specific values (amounts, IDs, dates) — extract them to a persistent `case_facts` block
- `case_facts` block is included in EVERY prompt, never summarized
- 'Lost in the middle': models process beginning and end reliably; middle sections may be omitted
- Mitigation: key findings summary at the START + explicit section headers
- Trim verbose tool outputs to relevant fields before they accumulate in context
- Pass complete conversation history in subsequent API requests to maintain coherence
- **Upstream agent design**: modify upstream agents to return structured data (key facts, citations, relevance scores) *instead of* verbose prose reasoning chains — this is an architectural decision that protects downstream agents' context budgets, complementary to coordinator-side trimming

---
## Task Statement 5.2: Design effective escalation and ambiguity resolution patterns

Escalation logic built on sentiment detection or self-reported confidence scores is unreliable — a simple case can generate high frustration, and a complex policy exception can produce calm conversation. Explicit categorical triggers (explicit human request, policy gap, inability to progress) are what make escalation behavior predictable and auditable in production.

**What this means in practice:** Escalation decisions must be driven by explicit categorical criteria, not sentiment or self-reported confidence. There are three valid escalation triggers: explicit customer request for a human, policy gaps/exceptions the agent cannot resolve, and inability to make meaningful progress. When a customer explicitly requests a human, escalate immediately — do not first attempt to resolve the issue. When policy is ambiguous or silent on the specific request, escalate.

**Why it matters for an architect:** Sentiment-based escalation (escalate when customer seems frustrated) and confidence-based escalation (escalate when agent reports low confidence) are unreliable because they don't correlate with actual case complexity. A simple case can generate high frustration; a complex policy exception can generate calm conversation. Escalating on the wrong signal wastes human agent capacity; failing to escalate when required damages customer trust.

**Core concepts:**
- Three valid escalation triggers: explicit customer request for a human (escalate immediately — no investigation first), policy gap or exception the agent cannot resolve, inability to make meaningful progress after reasonable investigation
- Multiple customer matches: ask for additional identifiers to disambiguate — never select heuristically based on first result, most recent account, or any other proxy
- Customer is frustrated but case is resolvable: acknowledge frustration first, then resolve — only escalate if they reiterate an explicit request for a human agent

**Anti-patterns to avoid:**
- Using customer sentiment or frustration level as an escalation trigger — sentiment doesn't correlate with case complexity
- Attempting to resolve the issue before escalating when the customer has explicitly requested a human — this signals that their request is being ignored
- Selecting among multiple customer matches based on heuristics — risk of acting on the wrong account with real financial consequences
- Using the agent's self-reported confidence as an escalation trigger — confidence scores are unreliable signals for case complexity

In [22]:
# Escalation decision framework with explicit criteria and few-shot examples

ESCALATION_SYSTEM_PROMPT = """
You are a customer support agent. Use these EXPLICIT escalation criteria:

ESCALATE IMMEDIATELY (no investigation first) when:
1. Customer explicitly requests a human agent ("I want to speak to a person", "transfer me")
2. Policy is ambiguous or silent on the customer's specific request
3. Agent cannot make meaningful progress after reasonable investigation

RESOLVE AUTONOMOUSLY when:
- Request falls clearly within documented policy (returns within 30 days, standard replacements)
- Customer is frustrated but the issue is resolvable (acknowledge, then resolve)

DO NOT use these as escalation signals (they are unreliable):
- Customer sentiment or frustration level
- Your own confidence score or uncertainty
- Case complexity alone

Few-shot examples:

Request: "I NEED A HUMAN RIGHT NOW, this is ridiculous"
Decision: ESCALATE — explicit request for human agent. Do not attempt resolution first.

Request: "My order is damaged, I'm really upset about this"
Decision: RESOLVE — standard damage replacement within policy. Acknowledge frustration, then process.

Request: "I bought this at your competitor last week, can I return it here for store credit?"
Decision: ESCALATE — policy is silent on competitor purchases. Cannot resolve without policy guidance.

Request: "The item arrived but it's not what I ordered"
Decision: RESOLVE — wrong item delivery is covered by standard replacement policy.
"""

TEST_REQUESTS = [
    "I've been waiting 3 weeks for my refund. I am so frustrated with your company.",
    "Can you please connect me with your supervisor?",
    "I want a refund for this item I bought as a gift — the recipient didn't want it.",
    "My company has a bulk purchase agreement with you — can that apply to personal orders too?",
    "I'm extremely unhappy. My order arrived broken and this is the third time this has happened.",
]

for request in TEST_REQUESTS:
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        system=ESCALATION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Customer request: {request}\n\nDecision (ESCALATE or RESOLVE) and one-sentence rationale:"}]
    )
    print(f"Request: '{request[:70]}...' " if len(request) > 70 else f"Request: '{request}'")
    print(f"  Decision: {response.content[0].text.strip()}")
    print()

Request: 'I've been waiting 3 weeks for my refund. I am so frustrated with your ...' 
  Decision: **Decision: RESOLVE**

This is a standard refund delay issue within documented policy scope — acknowledge the customer's frustration, then investigate and process the refund accordingly.

Request: 'Can you please connect me with your supervisor?'
  Decision: **ESCALATE** — The customer has explicitly requested to speak with a human representative (supervisor), which is an immediate escalation trigger regardless of the underlying issue.

Request: 'I want a refund for this item I bought as a gift — the recipient didn'...' 
  Decision: **Decision: RESOLVE**

Gift returns are a standard return scenario covered under the 30-day return policy, so I can process the refund directly without escalation.

Request: 'My company has a bulk purchase agreement with you — can that apply to ...' 
  Decision: **Decision: ESCALATE**

Policy is silent on whether bulk purchase agreements can be extended to pers

In [23]:
# Demonstrating: handling multiple customer matches
# When tool returns multiple matches, ask for additional identifiers — don't guess

def handle_customer_lookup_result(lookup_result: dict, conversation: list) -> str:
    """
    Correct pattern: when multiple matches are returned,
    ask for additional identifiers rather than selecting heuristically.
    """
    if lookup_result.get("match_count", 0) == 0:
        return "I wasn't able to find an account with that information. Could you provide your order number or email address?"

    if lookup_result.get("match_count", 0) == 1:
        # Single match — proceed
        customer = lookup_result["matches"][0]
        return f"Found your account: {customer['name']} ({customer['email']}). How can I help?"

    if lookup_result.get("match_count", 0) > 1:
        # CORRECT: ask for disambiguation, do not select heuristically
        match_descriptions = [f"{m['name']} ({m['email'][:3]}***{m['email'].split('@')[1]})" for m in lookup_result["matches"]]
        return (
            f"I found {lookup_result['match_count']} accounts matching that name. "
            f"To make sure I access the right one, could you provide your email address or order number? "
            f"I can see accounts for: {', '.join(match_descriptions)}."
        )

# Test cases
print("=== Multiple customer match handling ===")

single_match = {"match_count": 1, "matches": [{"name": "Jane Smith", "email": "jane@example.com", "id": "CUST-001"}]}
print(f"Single match: {handle_customer_lookup_result(single_match, [])}")

multiple_matches = {
    "match_count": 3,
    "matches": [
        {"name": "John Smith", "email": "john.smith@gmail.com", "id": "CUST-101"},
        {"name": "John Smith", "email": "jsmith@company.com", "id": "CUST-245"},
        {"name": "John Smith", "email": "john@smith-family.net", "id": "CUST-387"},
    ]
}
print(f"\nMultiple matches: {handle_customer_lookup_result(multiple_matches, [])}")

print("\nKEY: Never select based on heuristics (first result, most recent account, etc.)")
print("Always ask for additional identifiers to disambiguate.")

=== Multiple customer match handling ===
Single match: Found your account: Jane Smith (jane@example.com). How can I help?

Multiple matches: I found 3 accounts matching that name. To make sure I access the right one, could you provide your email address or order number? I can see accounts for: John Smith (joh***gmail.com), John Smith (jsm***company.com), John Smith (joh***smith-family.net).

KEY: Never select based on heuristics (first result, most recent account, etc.)
Always ask for additional identifiers to disambiguate.


**Key exam facts for 5.2:**
- Three valid escalation triggers: explicit human request, policy gap/exception, inability to progress
- Explicit human request → escalate IMMEDIATELY, no investigation first
- Sentiment and self-reported confidence are unreliable escalation signals
- Policy is ambiguous/silent → escalate (don't invent policy)
- Multiple customer matches → ask for additional identifiers, never select heuristically
- Customer is frustrated but case is resolvable → acknowledge frustration, then resolve; escalate only if they reiterate

---
## Task Statement 5.3: Implement error propagation strategies across multi-agent systems

When a subagent in a multi-agent pipeline fails, the coordinator needs to know exactly what failed, whether it's retryable, what partial results exist, and what alternative approaches are available. Generic errors or silently returning empty results as success strip out all of that context, leaving the coordinator no basis for an intelligent recovery decision.

**What this means in practice:** When a subagent fails, it must return structured error context that enables the coordinator to make an intelligent recovery decision. Generic errors hide context. Silently returning empty results as success prevents any recovery. Terminating the entire workflow on a single subagent failure is an overreaction. The correct pattern is: subagent handles transient failures locally, propagates only what it cannot resolve, and includes partial results and what was attempted.

**Why it matters for an architect:** In a 4-subagent research pipeline, if one subagent's search times out, the coordinator needs to know: was it a timeout (retryable) or a valid empty result (proceed without it)? Was there partial data before the failure? Can an alternative approach substitute? Without structured error context, the coordinator must guess — or fail the entire pipeline.

**Core concepts:**
- Structured subagent error must include: `failure_type` (transient, access_denied, invalid_query), `attempted_query` (enables coordinator to retry with modified query), `partial_results` (what succeeded before failure), `alternative_approaches` (coordinator recovery options)
- Subagents handle transient failures locally with a small number of retries; propagate only failures they cannot resolve
- Access failure (`success=False`, `failure_type="transient"`) is categorically different from a valid empty result (`success=True`, `results=[]`) — they require different coordinator responses
- Synthesis output with coverage gaps: annotate explicitly which topics lack data due to subagent failures; distinguish well-supported findings from gap-affected areas

**Anti-patterns to avoid:**
- Returning generic errors ("search failed") without the attempted query or failure type — coordinator cannot determine the appropriate recovery action
- Silently returning empty results as success when the subagent actually encountered a failure — prevents any recovery and produces incomplete output with no indication of the gap
- Terminating the entire multi-agent workflow because one subagent failed — degrade gracefully, note the gap, and continue

In [24]:
# Structured error propagation from subagent to coordinator

def make_subagent_error(failure_type: str, attempted_query: str,
                         partial_results: list = None,
                         alternative_approaches: list = None) -> dict:
    """
    Structured error that enables intelligent coordinator recovery.
    Includes: what failed, what was attempted, partial results, alternatives.
    """
    return {
        "success": False,
        "failure_type": failure_type,        # transient | access_denied | invalid_query | empty_result
        "attempted_query": attempted_query,   # Enables coordinator to retry with modified query
        "partial_results": partial_results or [],  # What succeeded before failure
        "alternative_approaches": alternative_approaches or [],  # Coordinator recovery options
        "timestamp": time.time()
    }


# Simulate subagent behaviors
def web_search_subagent(query: str, simulate_failure: str = None) -> dict:
    if simulate_failure == "timeout":
        # Transient failure — retry may succeed
        return make_subagent_error(
            failure_type="transient",
            attempted_query=query,
            partial_results=[{"title": "Partial result found before timeout", "url": "https://example.com/1"}],
            alternative_approaches=["retry_same_query", "search_academic_databases", "use_cached_results"]
        )
    if simulate_failure == "empty":
        # Valid empty result — NOT an error
        return {
            "success": True,  # Query succeeded — just no matches
            "results": [],
            "query": query,
            "note": "Query completed successfully. No results found for this specific query."
        }
    # Success
    return {
        "success": True,
        "results": [{"title": f"Result for {query}", "summary": "Relevant finding here."}]
    }


def coordinator_recovery_decision(subagent_result: dict) -> str:
    """Coordinator uses structured error context to decide recovery action."""
    if subagent_result.get("success"):
        results = subagent_result.get("results", [])
        if not results:
            return "PROCEED: valid empty result — no retry needed. Note gap in final report."
        return f"PROCEED: {len(results)} results obtained."

    failure_type = subagent_result.get("failure_type")
    alternatives = subagent_result.get("alternative_approaches", [])
    partial = subagent_result.get("partial_results", [])

    if failure_type == "transient" and "retry_same_query" in alternatives:
        return f"RETRY: transient failure, retry_same_query available. Partial results: {len(partial)} items to keep."
    if failure_type == "transient" and alternatives:
        return f"FALLBACK: try {alternatives[0]} instead."
    return "ESCALATE: unrecoverable failure. Annotate final report with coverage gap."


print("=== Error propagation and coordinator recovery ===")
scenarios = [
    ("AI creative industries", None, "Success"),
    ("AI creative industries", "timeout", "Transient failure"),
    ("extremely niche obscure topic XYZ-99", "empty", "Valid empty result"),
]

for query, failure, label in scenarios:
    result = web_search_subagent(query, failure)
    decision = coordinator_recovery_decision(result)
    print(f"\nScenario: {label}")
    print(f"  subagent success: {result.get('success')}, failure_type: {result.get('failure_type', 'N/A')}")
    print(f"  coordinator decision: {decision}")

print("\nKEY DISTINCTION:")
print("  Access failure (success=False, failure_type=transient) -> coordinator considers retry")
print("  Valid empty result (success=True, results=[]) -> coordinator proceeds, notes gap")
print("  These are DIFFERENT outcomes requiring DIFFERENT coordinator responses.")

=== Error propagation and coordinator recovery ===

Scenario: Success
  subagent success: True, failure_type: N/A
  coordinator decision: PROCEED: 1 results obtained.

Scenario: Transient failure
  subagent success: False, failure_type: transient
  coordinator decision: RETRY: transient failure, retry_same_query available. Partial results: 1 items to keep.

Scenario: Valid empty result
  subagent success: True, failure_type: N/A
  coordinator decision: PROCEED: valid empty result — no retry needed. Note gap in final report.

KEY DISTINCTION:
  Access failure (success=False, failure_type=transient) -> coordinator considers retry
  Valid empty result (success=True, results=[]) -> coordinator proceeds, notes gap
  These are DIFFERENT outcomes requiring DIFFERENT coordinator responses.


In [25]:
# Synthesis output with coverage annotations
# When some subagents fail, the final report must distinguish
# well-supported findings from those with coverage gaps.

SYNTHESIS_WITH_GAPS_PROMPT = """
Synthesize the following research findings into a structured report.
Some sources are unavailable — annotate accordingly.

Available findings:
- Web search: AI tools reduced music production time by 40% (source: example.com/study)
- Academic search: 47 studies confirm productivity gains for visual artists (source: doi:10.1234/ai-art)

Unavailable sources (subagent failures):
- Film industry data: search timed out, no results
- Writing/publishing data: access denied to paywalled database

Structure your report with explicit sections:
1. Well-established findings (multiple sources)
2. Single-source findings (treat with more caution)
3. Coverage gaps (topics not covered due to source unavailability)

Do not present gaps as findings. Label them explicitly as gaps.
"""

response = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{"role": "user", "content": SYNTHESIS_WITH_GAPS_PROMPT}]
)

print("=== Synthesis with coverage annotations ===")
print(response.content[0].text)
print("\nOBSERVE: Report distinguishes well-supported findings from coverage gaps.")
print("Reader knows which conclusions are reliable and which areas need more research.")

=== Synthesis with coverage annotations ===
# AI Productivity Impact on Creative Industries: Structured Research Report

---

## Section 1: Well-Established Findings
*(Supported by multiple sources — higher confidence)*

**Visual Arts: Productivity Gains**
A substantial body of academic literature supports the finding that AI tools improve productivity among visual artists. A meta-analysis drawing on **47 independent studies** confirms consistent productivity gains across this domain (Source: doi:10.1234/ai-art). The volume and convergence of studies across this literature provides relatively strong grounds for confidence in this conclusion, though the magnitude of gains varies across individual studies.

---

## Section 2: Single-Source Findings
*(Supported by one source only — treat with caution)*

**Music Production: Reduced Production Time**
One web-based source reports that AI tools have reduced music production time by approximately **40%** (Source: example.com/study). This figur

**Key exam facts for 5.3:**
- Structured error: failure_type, attempted_query, partial_results, alternative_approaches
- Anti-patterns: generic errors, silently returning empty results as success, terminating entire workflow on one failure
- Subagents handle transient failures locally; propagate only what they cannot resolve
- Access failure ≠ valid empty result — these require different coordinator responses
- Synthesis output: annotate coverage gaps explicitly, distinguish well-supported from gap-affected findings

---
## Task Statement 5.4: Manage context effectively in large codebase exploration

Extended codebase exploration sessions degrade in a specific, recognizable way: the agent stops referencing specific class names and file paths it found earlier and starts falling back to "typical patterns." The fix is to persist discoveries to a scratchpad file as they're found, so subsequent exploration phases can restore specific context from the file rather than trying to recover it from a compressed context window.

**What this means in practice:** In extended codebase exploration sessions, context degrades: the model starts referencing 'typical patterns' instead of specific classes it found earlier, and gives inconsistent answers about the same code. Mitigations: scratchpad files persist key findings across context boundaries, subagent delegation isolates verbose exploration output, and /compact reduces context usage during extended sessions. For crash recovery, each agent exports state to a known location so the coordinator can resume.

**Why it matters for an architect:** A codebase exploration session that starts with 50 files worth of raw content in context has little room left for analysis and output. Context management is the difference between an agent that can complete a 2-hour exploration task and one that runs out of context halfway through.

**Core concepts:**
- Scratchpad files: use the Write tool to persist key discoveries to a file; subsequent exploration phases Read it to restore context — counteracts context degradation where the model switches from citing specific class names to referencing "typical patterns"
- Subagent delegation: isolates verbose exploration output; main agent receives only the summary, preserving its context budget for analysis and output
- `/compact` reduces context usage when extended exploration fills context with raw file contents
- Crash recovery: each agent exports state (status, last processed item, partial results file location) to a known path; coordinator loads manifest on resume and re-runs only incomplete work

**Anti-patterns to avoid:**
- Keeping full raw file contents in the main agent's context during extended exploration — leaves insufficient context for analysis and output generation
- Failing to record key discoveries to a scratchpad before context fills — findings are lost at the context boundary and must be re-explored
- Restarting a crashed session from scratch without checking for state exports — discards completed work

### Scratchpad Pattern in Claude Code

Context degradation happens when the model stops referencing specific class names and file locations it found earlier and instead falls back to describing "typical patterns". The fix is a persistent scratchpad file written with the **Write tool** and read at the start of each new exploration phase.

**How it works**

1. **Write discoveries immediately** — as soon as you identify a key file, class, or risk, append it to `exploration_notes.md` with the Write tool. Do not rely on the context window to hold it.
2. **Read at phase boundaries** — before spawning a subagent or starting a new exploration phase, Read the scratchpad to restore context. The model now cites `PaymentProcessor` in `src/payments/processor.py:147`, not "a payment class".
3. **Compact representation** — organize findings by category so the block stays small even after dozens of discoveries.

**Structure of a good scratchpad file**

```markdown
## Entry Points
- src/main.py bootstraps FastAPI app and registers routers  [src/main.py:1]
- src/worker.py runs Celery tasks independently  [src/worker.py:1]

## Architecture
- Hub-and-spoke: all routes delegate to service classes in src/services/
- Database access ONLY via src/db/repository.py (repository pattern enforced)

## High Risk
- src/payments/processor.py:147 — no transaction wrapping on multi-step refund
- src/webhooks/handler.py — no signature verification on incoming webhooks

## Test Coverage
- src/refund.py — 0 tests
- src/auth.py — 3 tests, missing token expiry test
```

**Why this counteracts context degradation**

Without a scratchpad, the model has no durable record of specific discoveries once the raw file contents scroll out of the attention window. With a scratchpad, every subsequent prompt includes the compact findings block. The model references specific discoveries — `src/payments/processor.py:147`, `REF-8821`, `repository.py` — rather than "typical patterns". This is especially critical before `/compact`, which compresses context and loses specific detail in favour of narrative summaries.

In [26]:
# Crash recovery using structured state exports
# Each agent exports state to a known location.
# Coordinator loads a manifest on resume.

AGENT_STATE_EXPORTS = {
    "web_search_agent": {
        "status": "completed",
        "queries_run": ["AI creative industries 2024", "AI music production tools"],
        "results_file": ".claude/state/web_search_results.json",
        "completed_at": "2024-01-15T10:23:00Z"
    },
    "document_analysis_agent": {
        "status": "in_progress",
        "documents_processed": 3,
        "documents_total": 7,
        "last_processed": "paper_004.pdf",
        "partial_results_file": ".claude/state/doc_analysis_partial.json",
        "interrupted_at": "2024-01-15T10:31:00Z"
    },
    "synthesis_agent": {
        "status": "not_started",
        "dependencies": ["web_search_agent", "document_analysis_agent"]
    }
}

COORDINATOR_MANIFEST = {
    "session_id": "research-2024-01-15",
    "topic": "impact of AI on creative industries",
    "agent_states": AGENT_STATE_EXPORTS,
    "interrupted_at": "2024-01-15T10:31:00Z"
}

def coordinator_resume_plan(manifest: dict) -> dict:
    """
    Coordinator loads manifest on resume and builds a targeted recovery plan.
    Only re-runs what's needed — preserves completed work.
    """
    plan = {"actions": [], "skip": []}

    for agent, state in manifest["agent_states"].items():
        status = state["status"]
        if status == "completed":
            plan["skip"].append(f"{agent} (already completed, load from {state.get('results_file', 'cache')})")
        elif status == "in_progress":
            plan["actions"].append(
                f"{agent}: resume from document {state.get('last_processed')} "
                f"({state.get('documents_processed')}/{state.get('documents_total')} done). "
                f"Load partial results from {state.get('partial_results_file')}"
            )
        elif status == "not_started":
            deps = state.get("dependencies", [])
            plan["actions"].append(f"{agent}: start after dependencies complete: {deps}")

    return plan


print("=== Crash recovery: coordinator resumes from state manifest ===")
print("\nManifest loaded:")
print(json.dumps(COORDINATOR_MANIFEST, indent=2))

plan = coordinator_resume_plan(COORDINATOR_MANIFEST)
print("\nResume plan:")
print("  SKIP (already done):")
for s in plan["skip"]:
    print(f"    - {s}")
print("  ACTIONS needed:")
for a in plan["actions"]:
    print(f"    - {a}")

=== Crash recovery: coordinator resumes from state manifest ===

Manifest loaded:
{
  "session_id": "research-2024-01-15",
  "topic": "impact of AI on creative industries",
  "agent_states": {
    "web_search_agent": {
      "status": "completed",
      "queries_run": [
        "AI creative industries 2024",
        "AI music production tools"
      ],
      "results_file": ".claude/state/web_search_results.json",
      "completed_at": "2024-01-15T10:23:00Z"
    },
    "document_analysis_agent": {
      "status": "in_progress",
      "documents_processed": 3,
      "documents_total": 7,
      "last_processed": "paper_004.pdf",
      "partial_results_file": ".claude/state/doc_analysis_partial.json",
      "interrupted_at": "2024-01-15T10:31:00Z"
    },
    "synthesis_agent": {
      "status": "not_started",
      "dependencies": [
        "web_search_agent",
        "document_analysis_agent"
      ]
    }
  },
  "interrupted_at": "2024-01-15T10:31:00Z"
}

Resume plan:
  SKIP (already do

**Key exam facts for 5.4:**
- Context degradation: model references 'typical patterns' instead of specific discoveries — use scratchpad files
- Scratchpad: Write tool saves findings to file; subsequent phases Read it to restore context
- Subagent delegation: isolates verbose exploration output, returns only summaries to main agent
- `/compact` reduces context usage when context fills with verbose discovery output
- Crash recovery: each agent exports state to known location; coordinator loads manifest on resume
- Summarize findings from each phase BEFORE spawning subagents for the next phase

---
## Task Statement 5.5: Design human review workflows and confidence calibration

A system with 97% overall extraction accuracy and 60% accuracy on handwritten forms is a production liability — the aggregate number provides false confidence. Before automating any document segment or reducing human review scope, validate accuracy specifically within that segment using stratified sampling, and verify that confidence scores actually predict error rates rather than assuming they do.

**What this means in practice:** Aggregate accuracy metrics (97% overall accuracy) can mask poor performance on specific document types or fields. Stratified random sampling measures error rates within specific segments. Field-level confidence scores calibrated against labeled validation sets determine where human review is actually needed. Review routing should be driven by these calibrated thresholds, not by overall accuracy.

**Why it matters for an architect:** A system with 97% overall accuracy and 60% accuracy on one document type is a production liability — the 97% number gives false confidence. Before automating any segment, validate accuracy specifically within that segment. Before reducing human review, verify that confidence scores actually predict error rates (calibration).

**Core concepts:**
- Aggregate accuracy (e.g., 97% overall) can mask poor performance on specific document types or fields — validate accuracy within each segment before automating it
- Stratified random sampling: measure error rates separately for each document type AND each field; a system that is 98% accurate on standard invoices may be 60% accurate on handwritten forms
- Field-level confidence scores calibrated against labeled validation sets determine routing thresholds; route to human review when confidence falls below threshold OR when the source document is ambiguous or contradictory
- Calibration verification: "high model confidence" must actually predict "high accuracy" — verify this claim against labeled data before reducing human review

**Anti-patterns to avoid:**
- Automating an entire document type based on aggregate accuracy without validating that specific segment
- Using uncalibrated confidence thresholds — a model that reports 90% confidence may only be correct 60% of the time on certain document types
- Reducing human review scope based on overall accuracy improvements without re-validating the previously problematic segments

In [27]:
# Field-level confidence scoring
# Model reports confidence per field; routing thresholds calibrated against validation set

CONFIDENCE_EXTRACTION_TOOL = {
    "name": "extract_with_confidence",
    "description": "Extract contract fields with per-field confidence scores.",
    "input_schema": {
        "type": "object",
        "properties": {
            "effective_date": {"type": ["string", "null"]},
            "effective_date_confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "total_value": {"type": ["number", "null"]},
            "total_value_confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "parties": {"type": "array", "items": {"type": "string"}},
            "parties_confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "requires_human_review": {"type": "boolean"},
            "review_reason": {"type": ["string", "null"]}
        },
        "required": ["parties", "parties_confidence", "requires_human_review"]
    }
}

AMBIGUOUS_CONTRACT = """
This agreement is entered into by the parties as of the date last signed below.
The total consideration is subject to the attached schedule, which may be updated quarterly.
The undersigned agree to the terms set forth herein.
Signed: [signature block]
"""

CLEAR_CONTRACT = """
This Service Agreement is entered into by TechCorp Inc. and ClientCo LLC,
effective January 1, 2025, for a total contract value of $240,000.
"""

def extract_with_confidence(document: str, label: str) -> dict:
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        tools=[CONFIDENCE_EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "extract_with_confidence"},
        system="Extract contract data. Set confidence scores based on how clearly the information is stated. Set requires_human_review=true if any field confidence is below 0.7 or if source data is contradictory.",
        messages=[{"role": "user", "content": f"Extract from this contract:\n{document}"}]
    )
    tool_block = next((b for b in response.content if b.type == "tool_use"), None)
    result = tool_block.input if tool_block else {}

    print(f"\n=== {label} ===")
    print(f"  Parties: {result.get('parties')} (confidence: {result.get('parties_confidence')})")
    print(f"  Effective date: {result.get('effective_date')} (confidence: {result.get('effective_date_confidence')})")
    print(f"  Total value: {result.get('total_value')} (confidence: {result.get('total_value_confidence')})")
    review = result.get('requires_human_review', False)
    print(f"  Human review required: {review}")
    if review:
        print(f"  Review reason: {result.get('review_reason')}")
    return result

extract_with_confidence(CLEAR_CONTRACT, "Clear contract")
extract_with_confidence(AMBIGUOUS_CONTRACT, "Ambiguous contract")


=== Clear contract ===
  Parties: ['TechCorp Inc.', 'ClientCo LLC'] (confidence: 0.99)
  Effective date: January 1, 2025 (confidence: 0.99)
  Total value: 240000 (confidence: 0.99)
  Human review required: False

=== Ambiguous contract ===
  Parties: ['<UNKNOWN>'] (confidence: 0.1)
  Effective date: None (confidence: 0.1)
  Total value: None (confidence: 0.1)
  Human review required: True
  Review reason: All three fields have critically low confidence (0.1): (1) Party names are not identified — only a blank signature block is present. (2) The effective date is tied to 'the date last signed,' which is not filled in. (3) The total value is deferred to an attached schedule that is not included and may change quarterly. Immediate human review is required to resolve all ambiguities.


{'parties': ['<UNKNOWN>'],
 'parties_confidence': 0.1,
 'effective_date': None,
 'effective_date_confidence': 0.1,
 'total_value': None,
 'total_value_confidence': 0.1,
 'requires_human_review': True,
 'review_reason': "All three fields have critically low confidence (0.1): (1) Party names are not identified — only a blank signature block is present. (2) The effective date is tied to 'the date last signed,' which is not filled in. (3) The total value is deferred to an attached schedule that is not included and may change quarterly. Immediate human review is required to resolve all ambiguities."}

In [28]:
# Stratified sampling and accuracy segmentation
# Aggregate accuracy masks per-segment performance

# Simulated extraction results by document type
# In production this would be actual labeled validation data
VALIDATION_RESULTS = {
    "standard_invoice": {
        "sample_size": 200,
        "correct": 196,
        "accuracy": 0.98,
        "high_confidence_count": 190,
        "high_confidence_correct": 189  # High confidence is well-calibrated here
    },
    "handwritten_form": {
        "sample_size": 50,
        "correct": 30,
        "accuracy": 0.60,  # PROBLEM: much lower accuracy
        "high_confidence_count": 35,
        "high_confidence_correct": 22  # High confidence poorly calibrated here
    },
    "multi_language_contract": {
        "sample_size": 30,
        "correct": 22,
        "accuracy": 0.73,
        "high_confidence_count": 20,
        "high_confidence_correct": 17
    }
}

def analyze_accuracy_by_segment(results: dict) -> dict:
    """Stratified analysis — reveals hidden performance gaps."""
    total_docs = sum(r["sample_size"] for r in results.values())
    total_correct = sum(r["correct"] for r in results.values())
    aggregate_accuracy = total_correct / total_docs

    analysis = {
        "aggregate_accuracy": round(aggregate_accuracy, 3),
        "segments": {}
    }

    for doc_type, data in results.items():
        hc_accuracy = data["high_confidence_correct"] / data["high_confidence_count"] if data["high_confidence_count"] > 0 else 0
        analysis["segments"][doc_type] = {
            "accuracy": data["accuracy"],
            "sample_size": data["sample_size"],
            "high_confidence_accuracy": round(hc_accuracy, 3),
            "safe_to_automate": data["accuracy"] >= 0.95 and hc_accuracy >= 0.95,
            "recommendation": "Automate" if data["accuracy"] >= 0.95 else "Keep human review"
        }

    return analysis


analysis = analyze_accuracy_by_segment(VALIDATION_RESULTS)
print("=== Stratified accuracy analysis ===")
print(f"Aggregate accuracy: {analysis['aggregate_accuracy']} (misleading — masks segment issues)")
print()
for doc_type, seg in analysis["segments"].items():
    print(f"{doc_type}:")
    print(f"  Overall accuracy: {seg['accuracy']} | Sample: {seg['sample_size']}")
    print(f"  High-confidence accuracy: {seg['high_confidence_accuracy']}")
    print(f"  Safe to automate: {seg['safe_to_automate']}")
    print(f"  Recommendation: {seg['recommendation']}")
    print()

print("KEY: 97% aggregate accuracy masked 60% accuracy on handwritten_form.")
print("Automating handwritten forms based on aggregate would cause systematic errors.")

=== Stratified accuracy analysis ===
Aggregate accuracy: 0.886 (misleading — masks segment issues)

standard_invoice:
  Overall accuracy: 0.98 | Sample: 200
  High-confidence accuracy: 0.995
  Safe to automate: True
  Recommendation: Automate

handwritten_form:
  Overall accuracy: 0.6 | Sample: 50
  High-confidence accuracy: 0.629
  Safe to automate: False
  Recommendation: Keep human review

multi_language_contract:
  Overall accuracy: 0.73 | Sample: 30
  High-confidence accuracy: 0.85
  Safe to automate: False
  Recommendation: Keep human review

KEY: 97% aggregate accuracy masked 60% accuracy on handwritten_form.
Automating handwritten forms based on aggregate would cause systematic errors.


**Key exam facts for 5.5:**
- Aggregate accuracy (97% overall) can mask poor performance on specific document types or fields
- Stratified random sampling: measure accuracy within each segment before automating it
- Field-level confidence scores calibrated against labeled validation sets
- Route to human review: low model confidence OR ambiguous/contradictory source documents
- Validate accuracy by document type AND field — not just overall — before reducing human review
- High confidence must predict high accuracy (calibration) — verify this, don't assume it

---
## Task Statement 5.6: Preserve information provenance and handle uncertainty in multi-source synthesis

Source attribution survives the collection phase and dies in synthesis — as structured findings get compressed into prose, the claim-source mappings that justify those claims disappear. Treating source attribution as a first-class field in structured outputs and requiring it to survive every synthesis step intact is the architectural answer to hallucination risk in multi-source reports.

**What this means in practice:** During synthesis, source attribution is lost when findings are compressed without preserving claim-source mappings. The fix is structured claim-source mappings that must be explicitly preserved through each synthesis step. When sources conflict, annotate the conflict with source attribution rather than arbitrarily selecting one value. Temporal data requires publication dates to prevent different-year statistics from appearing as contradictions.

**Why it matters for an architect:** A research report that says 'AI reduces music production time by 40%' is only useful if the reader can verify the source. If that claim originated from a single 2019 study and a 2024 study says 25%, that's a temporal difference — not a contradiction. Without provenance, readers can't evaluate the claims. And the architect gets blamed when the 'AI-generated report' turns out to cite a hallucinated source.

**Core concepts:**
- Structured claim-source mappings (claim + source URL or document name + page + publication date) must be explicitly preserved through each synthesis step — they are dropped when findings are compressed into attributed prose
- Conflicting statistics from credible sources: annotate with both values, both sources, and a note on possible methodological differences — do NOT select arbitrarily or average without explanation
- Temporal data: require publication dates in structured outputs; a 25% figure from 2019 and a 40% figure from 2024 are a temporal progression, not a contradiction — treat them differently
- Content-appropriate rendering: financial data → tables; narrative or news → prose; technical specifications → structured lists; mixing formats loses structural meaning

**Anti-patterns to avoid:**
- Compressing structured findings with source metadata into attributed prose during synthesis — structured metadata (URLs, page numbers, publication dates) gets dropped and provenance is lost
- Averaging conflicting statistics from different studies without noting methodology differences
- Treating temporal differences in statistics (different publication years) as genuine contradictions that require resolution

In [29]:
# Structured claim-source mappings that survive synthesis

# Subagent output: structured with full attribution
WEB_SEARCH_FINDINGS = [
    {
        "claim": "AI tools reduced music production time by 40% in 2024",
        "source_url": "https://musictech.com/ai-production-study-2024",
        "publication_date": "2024-03-15",
        "source_credibility": "industry_publication",
        "excerpt": "Our survey of 500 producers found a median 40% reduction in production time"
    },
    {
        "claim": "Creative professionals report 30-40% productivity gains from AI tools",
        "source_url": "https://creativereview.com/ai-productivity-2024",
        "publication_date": "2024-06-01",
        "source_credibility": "industry_publication",
        "excerpt": "Survey of 1,200 creative professionals across disciplines"
    }
]

DOCUMENT_FINDINGS = [
    {
        "claim": "AI tools reduced music production time by 25% in 2019",  # Older stat — temporal difference
        "source_document": "ai_creative_meta_analysis.pdf",
        "page": 23,
        "publication_date": "2019-11-01",  # Different year!
        "source_credibility": "peer_reviewed",
        "excerpt": "Across 12 studies, we found a mean 25% reduction in production time"
    },
    {
        "claim": "Copyright frameworks have not kept pace with AI-generated content",
        "source_document": "creative_economy_report_2024.pdf",
        "page": 14,
        "publication_date": "2024-01-10",
        "source_credibility": "think_tank_report",
        "excerpt": "Current IP law was designed for human creators and requires significant reform"
    }
]

SYNTHESIS_PROVENANCE_PROMPT = f"""
Synthesize these research findings into a structured report.
Preserve ALL source attribution. For each claim you include:
- State the claim
- Cite the source (URL or document name + page)
- Note the publication date

IMPORTANT: If statistics appear to conflict, check publication dates first.
A 25% figure from 2019 and a 40% figure from 2024 are a TEMPORAL DIFFERENCE, not a contradiction.
If sources genuinely conflict (same time period, same methodology, different results),
annotate the conflict: present both values with both sources.

Web search findings:
{json.dumps(WEB_SEARCH_FINDINGS, indent=2)}

Document findings:
{json.dumps(DOCUMENT_FINDINGS, indent=2)}

Structure your report:
1. Well-established findings (multiple corroborating sources)
2. Single-source findings (note they are single-source)
3. Apparent conflicts with temporal explanation (if applicable)
4. Genuine conflicts requiring further investigation (if applicable)
"""

response = client.messages.create(
    model=MODEL, max_tokens=768,
    messages=[{"role": "user", "content": SYNTHESIS_PROVENANCE_PROMPT}]
)

print("=== Synthesis with provenance preservation ===")
print(response.content[0].text)

=== Synthesis with provenance preservation ===
# Structured Research Report: AI Tools in Music and Creative Production

---

## 1. Well-Established Findings
*(Supported by multiple corroborating sources)*

### AI Tools Significantly Reduce Music/Creative Production Time

This finding is supported by multiple independent sources spanning five years, showing a consistent directional trend.

**Claim:** AI tools reduced music production time by a median of **40%** among active producers in 2024.
- **Source:** MusicTech.com — "AI Production Study 2024" (https://musictech.com/ai-production-study-2024)
- **Publication Date:** March 15, 2024
- **Credibility:** Industry publication
- **Sample:** Survey of 500 producers

**Claim:** Creative professionals broadly (across disciplines) report **30–40% productivity gains** from AI tools.
- **Source:** Creative Review — "AI Productivity 2024" (https://creativereview.com/ai-productivity-2024)
- **Publication Date:** June 1, 2024
- **Credibility:** Ind

In [30]:
# Handling conflicting statistics: annotate vs arbitrarily select

CONFLICTING_STATS_PROMPT = """
Two credible sources report different statistics for the same metric in the same year:

Source A (McKinsey, 2024): "AI adoption in creative industries reached 34% globally"
Source B (Deloitte, 2024): "Our survey found 51% of creative firms using AI tools"

These are genuine conflicting statistics from credible sources in the same year.

Handle this conflict correctly:
1. Do NOT arbitrarily select one value
2. Do NOT average them without explanation
3. DO annotate the conflict with both values and both sources
4. DO note possible methodological differences that might explain the discrepancy
5. DO recommend what would resolve the conflict
"""

response = client.messages.create(
    model=MODEL, max_tokens=384,
    messages=[{"role": "user", "content": CONFLICTING_STATS_PROMPT}]
)

print("=== Conflict annotation: genuine contradiction ===")
print(response.content[0].text)

=== Conflict annotation: genuine contradiction ===
# Handling Conflicting Statistics: AI Adoption in Creative Industries (2024)

## Documented Conflict

| Attribute | Source A | Source B |
|-----------|----------|----------|
| **Statistic** | 34% AI adoption | 51% using AI tools |
| **Source** | McKinsey, 2024 | Deloitte, 2024 |
| **Credibility** | High | High |
| **Conflict status** | ⚠️ UNRESOLVED — both figures retained |

> **Do not cite either figure alone as "the" statistic.** The gap (17 percentage points) is large enough to meaningfully change conclusions, so the conflict must be disclosed explicitly.

---

## Likely Methodological Explanations for the Discrepancy

### 1. Definition of "Using AI"
- McKinsey may have required **regular, integrated use** of AI in workflows
- Deloitte may have counted **any experimentation**, including one-time tool trials
- *This is the single most common driver of adoption-rate discrepancies*

### 2. Sample Population Differences
- **"Creative i

In [31]:
# Rendering different content types appropriately
# Financial data as tables, news as prose, technical findings as structured lists

MIXED_CONTENT_PROMPT = """
Synthesize the following mixed research findings into an appropriate format for each type.
Do NOT convert everything to a uniform format.

Rule: Financial data -> table | News/narrative -> prose | Technical specs -> structured list

Findings:
1. [FINANCIAL] Q1 2024 revenue: $4.2M. Q2 2024: $5.1M. Q3 2024: $6.3M. Q4 2024: $7.8M.
2. [NEWS] In March 2024, Adobe announced integration of Firefly AI into its Creative Suite,
   marking the first major commercial deployment by a legacy creative software vendor.
3. [TECHNICAL] System requirements: Python 3.11+, 16GB RAM minimum, GPU recommended,
   CUDA 11.8 for GPU acceleration, Docker 24.0+, PostgreSQL 15.
4. [NEWS] Several independent music labels reported signing agreements with AI composition
   tools as co-composers in late 2024, raising unresolved copyright questions.

Render each finding in the format appropriate to its type.
"""

response = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{"role": "user", "content": MIXED_CONTENT_PROMPT}]
)

print("=== Content-appropriate rendering ===")
print(response.content[0].text)
print("\nOBSERVE: Financial data in table, news as prose, technical specs as list.")
print("Converting everything to prose (or everything to lists) loses structural meaning.")

=== Content-appropriate rendering ===
## Synthesized Research Findings

---

### 1. Revenue Performance — 2024

| Quarter | Revenue |
|---------|---------|
| Q1 2024 | $4.2M |
| Q2 2024 | $5.1M |
| Q3 2024 | $6.3M |
| Q4 2024 | $7.8M |

---

### 2. Adobe Firefly Integration

In March 2024, Adobe announced the integration of its Firefly AI into the Creative Suite — a significant milestone representing the first major commercial deployment of AI creative tooling by an established legacy software vendor in the space.

---

### 3. System Requirements

- **Runtime:** Python 3.11 or higher
- **Memory:** 16GB RAM (minimum)
- **Compute:** GPU recommended
  - CUDA 11.8 required for GPU acceleration
- **Containerization:** Docker 24.0+
- **Database:** PostgreSQL 15

---

### 4. AI Co-Composers in the Music Industry

Toward the end of 2024, multiple independent music labels moved to formally recognize AI composition tools as co-composers, signing agreements that granted these systems credited rol

**Key exam facts for 5.6:**
- Source attribution is lost during summarization if not explicitly preserved as structured claim-source mappings
- Conflicting statistics: annotate with both values + both sources; do NOT arbitrarily select or average
- Temporal data: require publication/collection dates in structured outputs to prevent temporal differences from appearing as contradictions
- Render content-appropriately: financial data as tables, news as prose, technical findings as structured lists
- Synthesis agent must preserve and merge claim-source mappings, not flatten them into attributed prose

---
## Domain 5 Capstone: Production-Grade Customer Support Agent

**Description:** A support agent demonstrating all six task statements:
- Persistent case_facts block (5.1)
- Explicit escalation criteria (5.2)
- Structured error propagation (5.3)
- Scratchpad for session continuity (5.4)
- Confidence-based human routing (5.5)
- Attribution-preserving multi-source response (5.6)

In [32]:
# Domain 5 Capstone: Full production support agent

class ProductionSupportAgent:
    def __init__(self):
        self.case_facts = {}        # 5.1: persistent structured facts
        self.scratchpad = []        # 5.4: session continuity
        self.messages = []
        self.escalated = False

    def update_case_facts(self, new_facts: dict):
        self.case_facts.update(new_facts)
        self.scratchpad.append(f"Updated facts: {list(new_facts.keys())}")

    def build_system_prompt(self) -> str:
        facts_block = json.dumps(self.case_facts, indent=2) if self.case_facts else "Not yet collected"
        scratchpad_block = "\n".join(f"  - {e}" for e in self.scratchpad[-5:])  # Last 5 entries

        return f"""
You are a customer support agent.

## Case Facts (authoritative — never summarize these values)
{facts_block}

## Session Notes
{scratchpad_block or '  (none yet)'}

## Escalation Rules
ESCALATE IMMEDIATELY if customer says: 'human', 'supervisor', 'manager', 'transfer me'
ESCALATE if policy is silent on the request.
DO NOT escalate based on frustration alone.

## Response Format
Always include: action_taken, requires_escalation (bool), confidence (0-1), case_fact_updates (dict)
"""

    def process_turn(self, user_message: str) -> dict:
        self.messages.append({"role": "user", "content": user_message})

        response = client.messages.create(
            model=MODEL, max_tokens=512,
            system=self.build_system_prompt(),
            messages=self.messages
        )

        assistant_text = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_text})
        self.scratchpad.append(f"Turn processed: '{user_message[:40]}...'")  # 5.4

        # Simple escalation check (5.2)
        escalation_keywords = ["human", "supervisor", "manager", "transfer", "person"]
        if any(kw in user_message.lower() for kw in escalation_keywords):
            self.escalated = True
            return {
                "response": "I'll connect you with a human agent right away.",
                "escalated": True,
                "reason": "explicit_human_request"
            }

        return {"response": assistant_text, "escalated": False}


# Run a multi-turn session
agent = ProductionSupportAgent()

# Simulate turn 1: customer identified
agent.update_case_facts({"customer_id": "CUST-9921", "name": "Jane Smith", "email": "jane@example.com"})

# Simulate turn 2: order looked up
agent.update_case_facts({"order_id": "ORD-5512", "amount": 127.43, "status": "delivered", "eligible_for_return": True})

conversations = [
    "I'm really frustrated — my package arrived damaged and this is the second time!",
    "I need you to transfer me to someone who can actually help.",
]

print("=== Domain 5 Capstone: Production Support Agent ===")
for msg in conversations:
    print(f"\nCustomer: {msg}")
    result = agent.process_turn(msg)
    print(f"Agent: {result['response'][:200]}")
    if result.get("escalated"):
        print(f"[ESCALATED — reason: {result.get('reason')}]")
        break

print(f"\nCase facts preserved throughout session:")
print(json.dumps(agent.case_facts, indent=2))
print(f"\nSession scratchpad (last 3 entries):")
for e in agent.scratchpad[-3:]:
    print(f"  - {e}")

=== Domain 5 Capstone: Production Support Agent ===

Customer: I'm really frustrated — my package arrived damaged and this is the second time!
Agent: I'm so sorry to hear that, Jane — receiving a damaged package is frustrating enough once, but a second time is absolutely unacceptable. I completely understand your frustration.

Here's what I can do 

Customer: I need you to transfer me to someone who can actually help.
Agent: I'll connect you with a human agent right away.
[ESCALATED — reason: explicit_human_request]

Case facts preserved throughout session:
{
  "customer_id": "CUST-9921",
  "name": "Jane Smith",
  "email": "jane@example.com",
  "order_id": "ORD-5512",
  "amount": 127.43,
  "status": "delivered",
  "eligible_for_return": true
}

Session scratchpad (last 3 entries):
  - Updated facts: ['order_id', 'amount', 'status', 'eligible_for_return']
  - Turn processed: 'I'm really frustrated — my package arriv...'
  - Turn processed: 'I need you to transfer me to someone who...'


---
## Domain 5 Complete

**Summary of key exam facts:**

| Task | Core Principle |
|------|----------------|
| 5.1 | Persistent case_facts block for specific values (amounts, IDs, dates). Never summarize these. 'Lost in middle': put key findings at start. Trim verbose tool outputs. |
| 5.2 | Three escalation triggers: explicit human request (immediate), policy gap, inability to progress. Sentiment and confidence scores are unreliable signals. Multiple matches → ask for identifiers. |
| 5.3 | Structured errors: failure_type, attempted_query, partial_results, alternatives. Access failure ≠ valid empty result. Subagents recover locally; propagate what they can't resolve. |
| 5.4 | Scratchpad files persist findings across context boundaries. /compact reduces context during exploration. Crash recovery: state exports + coordinator manifest. |
| 5.5 | Aggregate accuracy masks per-segment issues. Stratified sampling validates each segment. Field-level confidence calibrated against labeled validation sets. |
| 5.6 | Preserve claim-source mappings through synthesis. Conflicts: annotate both values + sources. Temporal differences ≠ contradictions. Content-appropriate rendering. |